# Phase 3 - Reproducible Multi-Month Ingestion & Provenance

Thin orchestration / narrative layer over `nhs_rtt.ingest` and
`nhs_rtt.crossmonth`. All reusable logic lives in `src/`; this notebook only
runs it and shows the evidence.

Architecture, the monthly acceptance contract and the resolution status of
**P2-U3 / P2-U5** (and the carry-forward of **P2-U7**) are documented in
`docs/phase3_ingestion.md`. Frozen Phase 1/2 contracts
(`validate_extract`, `load_rtt_csv`, `candidate_key_report`, the exact 105-band
schema) are **reused, not redefined**.

Raw files are opened read-only and never modified. This notebook does not
commit, tag or push anything.

In [1]:
from pathlib import Path

import pandas as pd

from nhs_rtt import ingest, crossmonth
from nhs_rtt.ingest import SourceRegistry, discover_sources
from nhs_rtt.crossmonth import run_phase3

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = REPO / "data" / "raw"
REGISTRY = RAW_DIR / "manifest.json"
OUT_DIR = REPO / "data" / "interim"
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
print(REPO)

C:\Users\abdul\OneDrive\Desktop\P_Projects\NHS-RTT-Waiting-Times


## 1. Deterministic source discovery

Only `rtt_YYYY_MM.csv` (exact, valid month) enters ingestion. Files that *look*
like an RTT monthly file but break the contract are surfaced as
**malformed candidates**; unrelated files are ignored. Output ordering never
depends on filesystem order or mtime.

In [2]:
disc = discover_sources(RAW_DIR)
print("production files     :", disc.production_files)
print("malformed candidates :", disc.malformed_candidates)
print("ignored (unrelated)  :", disc.ignored)
for m in disc.months:
    print(f"  {m.reporting_month}: {m.files}  multiple_candidates={m.multiple_candidates}")

production files     : ['rtt_2026_04.csv', 'rtt_2026_05.csv', 'rtt_2026_06.csv']
malformed candidates : []
ignored (unrelated)  : ['README.md', 'manifest.json']
  2026-04: ['rtt_2026_04.csv']  multiple_candidates=False
  2026-05: ['rtt_2026_05.csv']  multiple_candidates=False
  2026-06: ['rtt_2026_06.csv']  multiple_candidates=False


## 2. Source registry (the intended source set)

`data/raw/manifest.json` is the authoritative *intended* source set: reporting
period, filename, SHA-256, `selected` flag and an optional revision note. It is
tracked in Git. Generated validation results are **not** stored here.

In [3]:
reg = SourceRegistry.load(REGISTRY)
pd.DataFrame([
    {"reporting_period": e.reporting_period, "file": e.file,
     "sha256": e.sha256[:16] + "...", "selected": e.selected,
     "revision_note": e.revision_note}
    for e in reg.entries
])

,reporting_period,file,sha256,selected,revision_note
0,2026-04,rtt_2026_04.csv,0486aca5891a96af...,True,NaN
1,2026-05,rtt_2026_05.csv,fee364bc3654cf66...,True,NaN
2,2026-06,rtt_2026_06.csv,edc3927e4a006585...,True,Phase 1/2 development and validation month.


## 3. End-to-end run

`run_phase3` = reconcile the registry's **intended selected** months against
discovered + accepted sources (a missing intended month fails the run) ->
per-month acceptance (with revision resolution, month-bound provenance and
accepted-byte binding) -> cross-month diagnostics -> deterministic combine ->
**atomic** Parquet publication (staged, validated, then committed with a
generation marker) + data-level roundtrip.

In [4]:
result = run_phase3(raw_dir=RAW_DIR, registry_path=REGISTRY,
                    out_dir=OUT_DIR, write_parquet=True)
print(result.render_text())

=== Phase 3 - multi-month ingestion ===
raw dir            : C:\Users\abdul\OneDrive\Desktop\P_Projects\NHS-RTT-Waiting-Times\data\raw
production files   : ['rtt_2026_04.csv', 'rtt_2026_05.csv', 'rtt_2026_06.csv']
ignored (unrelated): ['README.md', 'manifest.json']
intended (selected): ['2026-04', '2026-05', '2026-06']
required this run  : ['2026-04', '2026-05', '2026-06']

--- per month ---
[2026-04] rtt_2026_04.csv
    sha256 observed : 0486aca5891a96af4f15e2f4559795138baef08b0ed580602b8c3a7aec6a56e9
    sha256 expected : 0486aca5891a96af4f15e2f4559795138baef08b0ed580602b8c3a7aec6a56e9 (registry)
    rows (loaded)   : 180781   streamed: 180781
    Period value    : ['RTT-April-2026']  -> canonical 2026-04
    105-band schema : OK  (bands=105)
    candidate key   : usable
    ACCEPTED        : True
[2026-05] rtt_2026_05.csv
    sha256 observed : fee364bc3654cf666f79d3485d26e2ada649a3418988a47aa1784e93294a5707
    sha256 expected : fee364bc3654cf666f79d3485d26e2ada649a3418988a47aa1784e

## 4. Per-month acceptance detail

Each accepted month passed: exact production filename, provenance (SHA-256 in
the registry), single internal `Period` that parses and agrees with the
filename, the frozen 105-band schema, a blank-preserving load, loaded rows ==
an independent streamed count, and a complete + unique candidate key.

In [5]:
rows = []
for month, r in sorted(result.acceptance.items()):
    rows.append({
        "month": month, "file": r.filename,
        "sha256_observed": r.sha256[:16] + "...",
        "sha256_matches_registry": r.sha256 == r.expected_sha256,
        "period_value": r.period_values[0] if r.period_values else None,
        "canonical_month": r.reporting_month,
        "rows_loaded": r.n_loaded_rows, "rows_streamed": r.n_data_rows,
        "schema_105_band_ok": r.schema_ok,
        "candidate_key_usable": r.candidate_key_usable,
        "accepted": r.accepted, "blocking": "; ".join(r.blocking),
        "warnings": "; ".join(r.warnings),
    })
pd.DataFrame(rows)

,month,file,sha256_observed,sha256_matches_registry,period_value,canonical_month,rows_loaded,rows_streamed,schema_105_band_ok,candidate_key_usable,accepted,blocking,warnings
0,2026-04,rtt_2026_04.csv,0486aca5891a96af...,True,RTT-April-2026,2026-04,180781,180781,True,True,True,,
1,2026-05,rtt_2026_05.csv,fee364bc3654cf66...,True,RTT-May-2026,2026-05,178171,178171,True,True,True,,
2,2026-06,rtt_2026_06.csv,edc3927e4a006585...,True,RTT-June-2026,2026-06,182411,182411,True,True,True,,


## 5. Cross-month mapping diagnostics (warnings, never rejection)

"Codes identify; names label." Changes are **surfaced, never auto-resolved**.
`<NA>` name groups are kept explicitly.

In [6]:
mapping = result.diagnostics["mapping"]
print("code -> name changes across months:")
display(mapping["code_name_changes"])
print("one name carried by several codes:")
display(mapping["name_code_collisions"])
print("codes appearing / disappearing between consecutive months:")
display(mapping["membership_changes"])

code -> name changes across months:


,dimension,code,months_present,n_distinct_names,names_by_month


one name carried by several codes:


,dimension,name,n_distinct_codes,codes,scope
0,provider,DUCHY HOSPITAL,2,"NT447,NVC04",within_month


codes appearing / disappearing between consecutive months:


,dimension,from_month,to_month,n_appeared,n_disappeared,appeared,disappeared
0,provider,2026-04,2026-05,4,1,"A9T5Y,D1V5N,P1N3Z,S5D8W",F2C9F
1,provider,2026-05,2026-06,5,0,"AW7,D3R0C,N3I9D,O1B1P,U1I3L",
2,commissioner,2026-05,2026-06,1,0,Y63,


## 6. Coverage & `Part_2A` subset diagnostics

Structural coverage and missingness prevalence per month, plus the frozen
`part_2a_subset_conformance` run for each month. `Part_2A > Part_2` rows are a
**source data-quality exception (P2-U7)**: preserved and flagged, never capped.

In [7]:
display(result.diagnostics["coverage"])
display(result.diagnostics["part2a_subset"])

,reporting_month,n_rows,n_treatment_function_codes,n_providers,n_commissioners,rows_Part_1A,rows_Part_1B,rows_Part_2,rows_Part_2A,rows_Part_3,pct_Total_blank,pct_unknown_clock_blank,pct_TotalAll_blank,pct_rows_all_bands_blank
0,2026-04,180781,24,529,128,18447,31614,63654,30541,36525,72.308,81.606,0.0,20.204
1,2026-05,178171,24,532,128,18351,30564,63321,30340,35595,72.546,81.729,0.0,19.978
2,2026-06,182411,24,537,129,18803,32351,63355,30548,37354,71.957,81.297,0.0,20.478


,reporting_month,n_part_2a_groups,n_without_matching_part_2,n_conformant,n_violations,violation_keys
0,2026-04,30541,5,30536,0,
1,2026-05,30340,3,30336,1,NT230/05V/C_100(2A=2>2=1)
2,2026-06,30548,0,30546,2,RTG/84H/C_502(2A=2>2=1); RTG/84H/C_999(2A=2>2=1)


## 7. Combined dataset & the CSV -> Parquet boundary

The combined frame keeps the **wide NHS source structure and raw values**; it
gains only four provenance columns (`source_file`, `source_sha256`,
`reporting_month`, `source_row_index` = the 0-based parsed data-row position
within its file). Row conservation and candidate-key completeness + uniqueness
are asserted inside `combine_months`.

In [8]:
sum_source_rows = sum(len(p.df) for p in result.payloads)
print("intended (selected) :", result.intended_months)
print("required this run   :", result.required_months,
      "  missing:", result.missing_intended, "  unregistered:", result.unexpected_months)
print("included months     :", [p.reporting_month for p in result.payloads])
print("combined rows       :", result.combined_rows, " == sum of sources:", sum_source_rows)
print("candidate key unique:", result.combined_key_unique)
print("parquet             :", result.parquet["parquet_path"])
print("parquet columns     :", result.parquet["columns"], "(121 source + 4 provenance)")
print("roundtrip (data)    :", result.roundtrip)
print("generation_id       :", result.parquet["generation_id"])
print("publication          :", result.publication)
print("verify_publication  :", crossmonth.verify_publication(OUT_DIR))
print("sidecar             :", result.parquet["sidecar_path"])
print("generation marker   :", result.parquet["generation_path"])
crossmonth.deterministic_manifest_view(result.parquet["manifest"])

intended (selected) : ['2026-04', '2026-05', '2026-06']
required this run   : ['2026-04', '2026-05', '2026-06']   missing: []   unregistered: []
included months     : ['2026-04', '2026-05', '2026-06']
combined rows       : 541363  == sum of sources: 541363
candidate key unique: True
parquet             : C:\Users\abdul\OneDrive\Desktop\P_Projects\NHS-RTT-Waiting-Times\data\interim\rtt_combined.parquet
parquet columns     : 125 (121 source + 4 provenance)
roundtrip (data)    : {'ok': True, 'problems': []}
generation_id       : f67b7342a30c4642a78864243142d910b84295dc7876ffb01c231cb8dc52627d
publication          : {'valid': True, 'generation_id': 'f67b7342a30c4642a78864243142d910b84295dc7876ffb01c231cb8dc52627d', 'combined_rows': 541363}
verify_publication  : {'valid': True, 'generation_id': 'f67b7342a30c4642a78864243142d910b84295dc7876ffb01c231cb8dc52627d', 'combined_rows': 541363}
sidecar             : C:\Users\abdul\OneDrive\Desktop\P_Projects\NHS-RTT-Waiting-Times\data\interim\rtt_co

{'artifact': 'rtt_combined.parquet',
 'deterministic': {'combined_rows': 541363,
  'column_count': 125,
  'provenance_columns': ['source_file',
   'source_sha256',
   'reporting_month',
   'source_row_index'],
  'candidate_key': ['Period',
   'Provider Org Code',
   'Commissioner Org Code',
   'RTT Part Type',
   'Treatment Function Code'],
  'candidate_key_unique': True,
  'candidate_key_usable': True,
  'months': [{'reporting_month': '2026-04',
    'source_file': 'rtt_2026_04.csv',
    'source_sha256': '0486aca5891a96af4f15e2f4559795138baef08b0ed580602b8c3a7aec6a56e9',
    'rows': 180781},
   {'reporting_month': '2026-05',
    'source_file': 'rtt_2026_05.csv',
    'source_sha256': 'fee364bc3654cf666f79d3485d26e2ada649a3418988a47aa1784e93294a5707',
    'rows': 178171},
   {'reporting_month': '2026-06',
    'source_file': 'rtt_2026_06.csv',
    'source_sha256': 'edc3927e4a0065855ad3b2347e7f82688e687b065406eefb49e2f67a9cd67f02',
    'rows': 182411}],
  'rows_by_month': {'2026-04': 18078

In [9]:
combined = pd.read_parquet(result.parquet["parquet_path"])
cols = ["Period", "Provider Org Code", "Commissioner Org Code", "RTT Part Type",
        "Treatment Function Code", "Total All", *crossmonth.PROVENANCE_COLS]
combined.loc[[0, 1, len(combined) // 2, len(combined) - 1], cols]

,Period,Provider Org Code,Commissioner Org Code,RTT Part Type,Treatment Function Code,Total All,source_file,source_sha256,reporting_month,source_row_index
0,RTT-April-2026,H3W7Q,06Q,Part_1A,C_130,12,rtt_2026_04.csv,0486aca5891a96af4f15e2f4559795138baef08b0ed580...,2026-04,0
1,RTT-April-2026,H3W7Q,06Q,Part_1A,C_999,12,rtt_2026_04.csv,0486aca5891a96af4f15e2f4559795138baef08b0ed580...,2026-04,1
270681,RTT-May-2026,RHM,02T,Part_3,X02,1,rtt_2026_05.csv,fee364bc3654cf666f79d3485d26e2ada649a3418988a4...,2026-05,89900
541362,RTT-June-2026,U6A5N,W2U3Z,Part_3,C_999,2,rtt_2026_06.csv,edc3927e4a0065855ad3b2347e7f82688e687b065406ee...,2026-06,182410


In [10]:
# missingness is preserved through the Parquet boundary (not zero-filled)
band_cols = [c for c in combined.columns if c.startswith("Gt ")]
print("rows with every band blank :", int(combined[band_cols].isna().all(axis=1).sum()))
print("Total blank                :", int(combined["Total"].isna().sum()), "/", len(combined))
print("Total All blank             :", int(combined["Total All"].isna().sum()))
print("dtype Total / source_row_index:", combined["Total"].dtype, "/", combined["source_row_index"].dtype)

rows with every band blank : 109474
Total blank                : 391233 / 541363
Total All blank             : 0
dtype Total / source_row_index: Int64 / Int64


## 8. Cross-month findings vs frozen Phase 2

| Check | Result (April - June 2026) |
|---|---|
| 105-band schema drift | none - identical 121-col header, all pass `validate_extract` |
| `Period` format | `RTT-<MonthName>-<YYYY>`, one value per file, agrees with filename |
| Candidate key | unique + usable per month **and** combined |
| Code <-> name maps | stable across the three months (no drift) |
| RTT part structure | all five parts present each month |
| Treatment-function coverage | 24 codes each month, identical set |
| `Part_2A > Part_2` exceptions | **P2-U7 carried**: April 0, May 1 (`NT230`/`05V`/`C_100`), June 2 (`RTG`/`84H`) - preserved & flagged |
| Provider / commissioner membership | minor churn (new/retired codes) - warning only |
| Contradiction to a frozen Phase 2 decision | **none** |

### Post-audit remediation guarantees (P3-A01-A07)

* **Intended-source completeness** - the registry's selected months are
  reconciled against discovered + accepted sources; a missing intended month
  fails the run and does **not** re-publish a reduced set.
* **Month-bound provenance** - a registry digest authorises a source only for
  its own reporting month; a cross-month hash match is rejected.
* **Accepted-byte binding** - the frame published for a month is the one loaded
  during acceptance and bound to its digest; the source path is never reopened.
* **Publication integrity** - Parquet + sidecar + generation marker are staged,
  validated, then committed (marker last); a pair is valid iff
  `verify_publication` confirms the marker's hashes match both files.
* **Reserved provenance names** - a source carrying `source_file` /
  `source_sha256` / `reporting_month` / `source_row_index` is rejected.
* Backup/temp-suffixed RTT names are surfaced as malformed candidates; malformed
  external inputs produce structured rejections, not raw `TypeError`s.

**P2-U3** (cross-month stability of schema / candidate key / code-name maps;
revised-release handling) and **P2-U5** (`Period` parsing + filename agreement +
revised-release policy) are **addressed** for ingestion. Month-end *date*
derivation and the stock/flow analytical temporal model stay Phase 4/5.
**P2-U7** remains an open, flagged source data-quality item (preserve & flag).

Phase 3 leaves the data recognisably in source structure plus provenance - no
reshaping, cleaning, modelling or KPIs (those are Phase 4+).